# 🔍 MRAG Pipeline — Multimodal RAG (100% Free)

**Each extracted element is saved separately to Google Drive:**
```
MyDrive/MRAG_Output/
  ├── images/   → page_1_img_0.png,  page_2_img_0.png  ...
  ├── tables/   → page_1_table_0.csv, page_3_table_1.csv ...
  └── texts/    → page_1_text.txt,   page_2_text.txt   ...
```

**Full pipeline:**
```
PDF → Extract & Save to Drive
        ├─ images/ → BLIP caption  → summary ─┐
        ├─ tables/ → BART summary  → summary ──┼─→ MiniLM embed → ChromaDB
        └─ texts/  → BART summary  → summary ─┘
                                                       │
             Raw elements ──────────────→ SQLite   ←──┘ doc_id
                                                        │
              Query → embed → top-k ids ────────────────┘ → flan-t5 → Answer
```

| Component | Model | Cost |
|---|---|---|
| PDF extraction | `pdfplumber` | Free |
| Text/Table summary | `facebook/bart-large-cnn` | Free |
| Image captioning | `Salesforce/blip-image-captioning-large` | Free |
| Answer generation | `google/flan-t5-large` | Free |
| Embeddings | `all-MiniLM-L6-v2` | Free |
| Vector DB | ChromaDB (local) | Free |
| Document Store | SQLite | Free |

> ⚡ **Enable GPU:** Runtime → Change runtime type → T4 GPU

## 📦 Step 1 — Mount Google Drive & Install Dependencies

In [30]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [31]:
!pip install -q pdfplumber transformers torch accelerate \
                sentence-transformers chromadb Pillow
print('✅ Dependencies installed')

✅ Dependencies installed


## ⚙️ Step 2 — Imports & Config

In [32]:
import uuid, json, base64, sqlite3, logging, io, csv
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

import torch
import chromadb
import pdfplumber
from PIL import Image
from sentence_transformers import SentenceTransformer
from transformers import (
    pipeline,
    BlipProcessor,
    BlipForConditionalGeneration,
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
)

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
log = logging.getLogger(__name__)

# ── Free HuggingFace models ──────────────────────────────────────
SUMMARY_MODEL   = 'facebook/bart-large-cnn'
QA_MODEL        = 'google/flan-t5-large'
VISION_MODEL    = 'Salesforce/blip-image-captioning-large'
EMBEDDING_MODEL = 'all-MiniLM-L6-v2'

# ── Google Drive output folders ──────────────────────────────────
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/content'
DRIVE_IMAGES_DIR = f'{DRIVE_OUTPUT_DIR}/images'
DRIVE_TABLES_DIR = f'{DRIVE_OUTPUT_DIR}/tables'
DRIVE_TEXTS_DIR  = f'{DRIVE_OUTPUT_DIR}/texts'

# ── Local storage (ChromaDB + SQLite) ───────────────────────────
CHROMA_DIR  ='/content/drive/MyDrive/content'
SQLITE_PATH = '/content/drive/MyDrive/content/document_store.db' # Modified to include database file name
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Using device: {DEVICE}')

✅ Using device: cuda


## 📁 Step 3 — Create Drive Output Folders

In [33]:
for folder in [DRIVE_IMAGES_DIR, DRIVE_TABLES_DIR, DRIVE_TEXTS_DIR]:
    Path(folder).mkdir(parents=True, exist_ok=True)
    print(f'📁 {folder}')

print('\n✅ Drive folders ready')

📁 /content/drive/MyDrive/content/images
📁 /content/drive/MyDrive/content/tables
📁 /content/drive/MyDrive/content/texts

✅ Drive folders ready


## 🗄️ Step 4 — Data Model & Document Store (SQLite)

In [34]:
@dataclass
class Element:
    """A single extracted element from a PDF page."""
    doc_id:        str
    element_type:  str   # 'text' | 'table' | 'image'
    page_number:   int
    raw_content:   str   # plain text, markdown table, or base64 PNG
    summary:       str = ''
    drive_path:    str = ''   # path where file was saved on Drive
    metadata:      dict = field(default_factory=dict)


class DocumentStore:
    """SQLite store — holds all elements with their Drive file path."""

    def __init__(self, db_path: str = SQLITE_PATH):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS elements (
                doc_id        TEXT PRIMARY KEY,
                element_type  TEXT NOT NULL,
                page_number   INTEGER,
                raw_content   TEXT,
                summary       TEXT,
                drive_path    TEXT,
                metadata      TEXT
            )
        """)
        self.conn.commit()
        log.info('DocumentStore ready at %s', db_path)

    def upsert(self, element: Element):
        self.conn.execute("""
            INSERT OR REPLACE INTO elements
              (doc_id, element_type, page_number, raw_content, summary, drive_path, metadata)
            VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (
            element.doc_id, element.element_type, element.page_number,
            element.raw_content, element.summary, element.drive_path,
            json.dumps(element.metadata),
        ))
        self.conn.commit()

    def get(self, doc_id: str) -> Optional[Element]:
        row = self.conn.execute(
            'SELECT * FROM elements WHERE doc_id = ?', (doc_id,)
        ).fetchone()
        if not row:
            return None
        return Element(
            doc_id=row[0], element_type=row[1], page_number=row[2],
            raw_content=row[3], summary=row[4], drive_path=row[5],
            metadata=json.loads(row[6]),
        )

    def close(self):
        self.conn.close()

print('✅ DocumentStore defined')

✅ DocumentStore defined


## 💾 Step 5 — Drive Saver (saves each element type to its own folder)

In [35]:
class DriveSaver:
    """
    Saves each extracted element to the correct Drive folder:
      images/ → .png files
      tables/ → .csv files
      texts/  → .txt files
    Returns the saved file path.
    """

    # Track per-page counters so filenames don't collide
    _counters: dict = {}

    @classmethod
    def _next(cls, key: str) -> int:
        cls._counters[key] = cls._counters.get(key, -1) + 1
        return cls._counters[key]

    @classmethod
    def save(cls, element: Element) -> str:
        page = element.page_number
        if element.element_type == 'image':
            return cls._save_image(element.raw_content, page)
        elif element.element_type == 'table':
            return cls._save_table(element.raw_content, page)
        elif element.element_type == 'text':
            return cls._save_text(element.raw_content, page)
        return ''

    @classmethod
    def _save_image(cls, img_b64: str, page: int) -> str:
        idx  = cls._next(f'img_{page}')
        path = f'{DRIVE_IMAGES_DIR}/page_{page}_img_{idx}.png'
        img  = Image.open(io.BytesIO(base64.b64decode(img_b64))).convert('RGB')
        img.save(path)
        log.info('    💾 Saved image  → %s', path)
        return path

    @classmethod
    def _save_table(cls, table_md: str, page: int) -> str:
        idx  = cls._next(f'tbl_{page}')
        path = f'{DRIVE_TABLES_DIR}/page_{page}_table_{idx}.csv'
        # Convert markdown table back to CSV
        lines = [l for l in table_md.strip().split('\n') if not l.startswith('| ---')]
        with open(path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            for line in lines:
                row = [c.strip() for c in line.strip('|').split('|')]
                writer.writerow(row)
        log.info('    💾 Saved table  → %s', path)
        return path

    @classmethod
    def _save_text(cls, text: str, page: int) -> str:
        idx  = cls._next(f'txt_{page}')
        path = f'{DRIVE_TEXTS_DIR}/page_{page}_text_{idx}.txt'
        Path(path).write_text(text, encoding='utf-8')
        log.info('    💾 Saved text   → %s', path)
        return path

print('✅ DriveSaver defined')

✅ DriveSaver defined


## 📄 Step 6 — PDF Extractor

In [36]:
class PDFExtractor:
    """Extracts text, tables, and images from every page of a PDF."""

    def extract(self, pdf_path: str) -> list[Element]:
        elements = []
        log.info('Extracting: %s', pdf_path)

        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages, start=1):

                # ── Text ──────────────────────────────────────────
                text = page.extract_text() or ''
                if text.strip():
                    elements.append(Element(
                        doc_id=str(uuid.uuid4()), element_type='text',
                        page_number=page_num, raw_content=text,
                        metadata={'source': str(pdf_path)},
                    ))

                # ── Tables ────────────────────────────────────────
                for tbl in page.extract_tables():
                    if tbl:
                        elements.append(Element(
                            doc_id=str(uuid.uuid4()), element_type='table',
                            page_number=page_num, raw_content=self._to_markdown(tbl),
                            metadata={'source': str(pdf_path)},
                        ))

                # ── Images ────────────────────────────────────────
                for img_obj in page.images:
                    try:
                        b64 = self._extract_b64(pdf, img_obj, page_num)
                        if b64:
                            elements.append(Element(
                                doc_id=str(uuid.uuid4()), element_type='image',
                                page_number=page_num, raw_content=b64,
                                metadata={'source': str(pdf_path)},
                            ))
                    except Exception as exc:
                        log.warning('Page %d image skip: %s', page_num, exc)

        log.info('Extracted %d elements total', len(elements))
        return elements

    @staticmethod
    def _to_markdown(rows: list) -> str:
        if not rows: return ''
        header = '| ' + ' | '.join(str(c or '') for c in rows[0]) + ' |'
        sep    = '| ' + ' | '.join('---' for _ in rows[0]) + ' |'
        body   = '\n'.join('| ' + ' | '.join(str(c or '') for c in row) + ' |' for row in rows[1:])
        return '\n'.join([header, sep, body])

    @staticmethod
    def _extract_b64(pdf, img_obj, page_num) -> Optional[str]:
        try:
            page = pdf.pages[page_num - 1]
            bbox = (img_obj['x0'], img_obj['top'], img_obj['x1'], img_obj['bottom'])
            buf  = io.BytesIO()
            page.within_bbox(bbox).to_image(resolution=150).save(buf, format='PNG')
            return base64.b64encode(buf.getvalue()).decode('utf-8')
        except Exception:
            return None

print('✅ PDFExtractor defined')

✅ PDFExtractor defined


## 🤖 Step 7 — Model Manager (lazy loading)

In [37]:
class ModelManager:
    """Lazily loads HuggingFace models once and caches them."""
    _summariser = None
    _qa         = None
    _blip_proc  = None
    _blip_model = None
    _embedder   = None

    @classmethod
    def summariser(cls):
        if cls._summariser is None:
            log.info('Loading summarisation model: %s', SUMMARY_MODEL)
            cls._summariser = pipeline(
                'summarization', model=SUMMARY_MODEL,
                device=0 if DEVICE == 'cuda' else -1,
            )
        return cls._summariser

    @classmethod
    def qa_tokenizer_model(cls):
        if cls._qa is None:
            log.info('Loading QA model: %s', QA_MODEL)
            tok   = AutoTokenizer.from_pretrained(QA_MODEL)
            model = AutoModelForSeq2SeqLM.from_pretrained(QA_MODEL).to(DEVICE)
            cls._qa = (tok, model)
        return cls._qa

    @classmethod
    def blip(cls):
        if cls._blip_proc is None:
            log.info('Loading vision model: %s', VISION_MODEL)
            cls._blip_proc  = BlipProcessor.from_pretrained(VISION_MODEL)
            cls._blip_model = BlipForConditionalGeneration.from_pretrained(VISION_MODEL).to(DEVICE)
        return cls._blip_proc, cls._blip_model

    @classmethod
    def embedder(cls):
        if cls._embedder is None:
            log.info('Loading embedding model: %s', EMBEDDING_MODEL)
            cls._embedder = SentenceTransformer(EMBEDDING_MODEL)
        return cls._embedder

print('✅ ModelManager defined')

✅ ModelManager defined


## ✍️ Step 8 — Summariser (BART + BLIP)

In [38]:
class Summariser:
    """text/table → BART   |   image → BLIP"""

    def summarise(self, element: Element) -> str:
        try:
            if element.element_type == 'text':
                return self._summarise_text(element.raw_content)
            elif element.element_type == 'table':
                return self._summarise_table(element.raw_content)
            elif element.element_type == 'image':
                return self._caption_image(element.raw_content)
        except Exception as exc:
            log.error('Summarise failed [%s]: %s', element.doc_id, exc)
        return ''

    @staticmethod
    def _summarise_text(text: str) -> str:
        result = ModelManager.summariser()(
            text[:3000], max_length=130, min_length=30, do_sample=False
        )
        return result[0]['summary_text']

    @staticmethod
    def _summarise_table(table_md: str) -> str:
        prompt = f'Summarise this table:\n{table_md[:3000]}'
        result = ModelManager.summariser()(
            prompt[:1024], max_length=130, min_length=20, do_sample=False
        )
        return result[0]['summary_text']

    @staticmethod
    def _caption_image(img_b64: str) -> str:
        image            = Image.open(io.BytesIO(base64.b64decode(img_b64))).convert('RGB')
        processor, model = ModelManager.blip()
        inputs           = processor(image, return_tensors='pt').to(DEVICE)
        out              = model.generate(**inputs, max_new_tokens=100)
        return processor.decode(out[0], skip_special_tokens=True)

print('✅ Summariser defined')

✅ Summariser defined


## 🗂️ Step 9 — Vector Store (ChromaDB + MiniLM)

In [39]:
class VectorStore:
    """Stores summary embeddings in ChromaDB using all-MiniLM-L6-v2."""

    def __init__(self, persist_dir: str = CHROMA_DIR):
        self.client     = chromadb.PersistentClient(path=persist_dir)
        self.collection = self.client.get_or_create_collection(
            name='mrag_summaries', metadata={'hnsw:space': 'cosine'}
        )
        log.info('VectorStore ready at %s', persist_dir)

    def add(self, element: Element):
        if not element.summary:
            return
        embedding = ModelManager.embedder().encode(element.summary).tolist()
        self.collection.upsert(
            ids=[element.doc_id],
            embeddings=[embedding],
            documents=[element.summary],
            metadatas=[{
                'element_type': element.element_type,
                'page_number':  element.page_number,
                'drive_path':   element.drive_path,
                **element.metadata,
            }],
        )

    def query(self, query_text: str, top_k: int = 5) -> list[dict]:
        embedding = ModelManager.embedder().encode(query_text).tolist()
        n         = min(top_k, self.collection.count())
        if n == 0: return []
        results = self.collection.query(query_embeddings=[embedding], n_results=n)
        return [
            {
                'doc_id':   results['ids'][0][i],
                'summary':  results['documents'][0][i],
                'distance': results['distances'][0][i],
                'metadata': results['metadatas'][0][i],
            }
            for i in range(len(results['ids'][0]))
        ]

print('✅ VectorStore defined')

✅ VectorStore defined


## 🔄 Step 10 — Ingestion Pipeline (Extract → Save to Drive → Summarise → Index)

In [40]:
class IngestionPipeline:
    """
    Full ingestion flow:
      PDF
       → Extract elements
       → Save each to Drive (images/ tables/ texts/)
       → Summarise
       → Store raw + drive path in SQLite
       → Store summary embedding in ChromaDB
    """

    def __init__(self):
        self.extractor    = PDFExtractor()
        self.drive_saver  = DriveSaver()
        self.summariser   = Summariser()
        self.doc_store    = DocumentStore()
        self.vector_store = VectorStore()

    def ingest(self, pdf_path: str):
        log.info('═══ INGESTION START: %s ═══', pdf_path)
        elements = self.extractor.extract(pdf_path)

        counts = {'text': 0, 'table': 0, 'image': 0}

        for idx, elem in enumerate(elements, 1):
            log.info('  [%d/%d] Processing %s (page %d)…',
                     idx, len(elements), elem.element_type, elem.page_number)

            # 1. Save raw file to correct Drive folder
            elem.drive_path = self.drive_saver.save(elem)

            # 2. Generate summary
            elem.summary = self.summariser.summarise(elem)

            # 3. Store in Document Store (SQLite)
            self.doc_store.upsert(elem)

            # 4. Store embedding in Vector DB (ChromaDB)
            self.vector_store.add(elem)

            counts[elem.element_type] += 1

        log.info('═══ INGESTION COMPLETE ═══')
        print(f'\n✅ Indexed {len(elements)} elements:')
        print(f'   📝 Texts  : {counts["text"]}  → {DRIVE_TEXTS_DIR}')
        print(f'   📊 Tables : {counts["table"]}  → {DRIVE_TABLES_DIR}')
        print(f'   🖼️  Images : {counts["image"]}  → {DRIVE_IMAGES_DIR}')

    def close(self):
        self.doc_store.close()

print('✅ IngestionPipeline defined')

✅ IngestionPipeline defined


## 🔎 Step 11 — Retrieval Pipeline

In [41]:
class RetrievalPipeline:
    """Query → Embed → ChromaDB search → Fetch from SQLite → flan-t5 answer."""

    def __init__(self):
        self.doc_store    = DocumentStore()
        self.vector_store = VectorStore()

    def retrieve(self, query: str, top_k: int = 5) -> list[Element]:
        log.info("Retrieving top-%d for: '%s'", top_k, query)
        hits     = self.vector_store.query(query, top_k=top_k)
        elements = []
        for hit in hits:
            elem = self.doc_store.get(hit['doc_id'])
            if elem:
                log.info('  • [%.3f] %s  page=%d  drive=%s',
                         hit['distance'], elem.element_type,
                         elem.page_number, elem.drive_path)
                elements.append(elem)
        return elements

    def answer(self, query: str, top_k: int = 5) -> str:
        elements = self.retrieve(query, top_k=top_k)
        if not elements:
            return 'No relevant content found in the document.'

        context_parts = []
        for elem in elements:
            label = f'[{elem.element_type.upper()} page {elem.page_number}]'
            if elem.element_type == 'image':
                context_parts.append(f'{label}: {elem.summary}')
            else:
                context_parts.append(f'{label}:\n{elem.raw_content[:800]}')

        prompt = (
            'Answer the question based on the context below.\n\n'
            'Context:\n' + '\n\n'.join(context_parts)[:3500] +
            f'\n\nQuestion: {query}\nAnswer:'
        )

        tokenizer, model = ModelManager.qa_tokenizer_model()
        inputs  = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to(DEVICE)
        outputs = model.generate(**inputs, max_new_tokens=300)
        return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    def close(self):
        self.doc_store.close()

print('✅ RetrievalPipeline defined')

✅ RetrievalPipeline defined


## 📥 Step 12 — Ingest Your PDF

In [42]:
PDF_PATH = '/content/drive/MyDrive/content/attention-is-all-you-need.pdf'

ingest_pipe = IngestionPipeline()
ingest_pipe.ingest(PDF_PATH)
ingest_pipe.close()

ERROR:__main__:Summarise failed [86b7c59a-873b-4549-8c43-9c99b005e438]: "Unknown task summarization, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'image-to-image', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'question-answering', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'visual-question-answering', 'vqa', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection', 'translation_XX_to_YY']"
ERROR:__main__:Summarise failed [63bf4007-6c38-4109-9740-bb22859e4666]: "Unknown task summarization, available tasks are ['any-to-any', 'audio-classification

Loading weights:   0%|          | 0/616 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-large
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
ERROR:__main__:Summarise failed [a19ac0dc-4c46-4910-8867-d503a146c60b]: "Unknown task summarization, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'image-to-image', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'question-answering', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'visual-question-answering'


✅ Indexed 28 elements:
   📝 Texts  : 15  → /content/drive/MyDrive/content/texts
   📊 Tables : 10  → /content/drive/MyDrive/content/tables
   🖼️  Images : 3  → /content/drive/MyDrive/content/images


## 📂 Step 13 — Verify Files Saved to Drive

In [43]:
import os

for folder, label in [
    (DRIVE_TEXTS_DIR,  '📝 texts/'),
    (DRIVE_TABLES_DIR, '📊 tables/'),
    (DRIVE_IMAGES_DIR, '🖼️  images/'),
]:
    files = sorted(os.listdir(folder))
    print(f'\n{label}  ({len(files)} files)')
    for f in files:
        size = os.path.getsize(f'{folder}/{f}')
        print(f'  {f}  ({size:,} bytes)')


📝 texts/  (15 files)
  page_10_text_0.txt  (2,804 bytes)
  page_11_text_0.txt  (2,976 bytes)
  page_12_text_0.txt  (2,952 bytes)
  page_13_text_0.txt  (787 bytes)
  page_14_text_0.txt  (781 bytes)
  page_15_text_0.txt  (779 bytes)
  page_1_text_0.txt  (2,607 bytes)
  page_2_text_0.txt  (3,780 bytes)
  page_3_text_0.txt  (1,616 bytes)
  page_4_text_0.txt  (2,225 bytes)
  page_5_text_0.txt  (2,867 bytes)
  page_6_text_0.txt  (3,077 bytes)
  page_7_text_0.txt  (2,937 bytes)
  page_8_text_0.txt  (2,782 bytes)
  page_9_text_0.txt  (2,696 bytes)

📊 tables/  (10 files)
  page_10_table_0.csv  (249 bytes)
  page_13_table_0.csv  (521 bytes)
  page_13_table_1.csv  (3 bytes)
  page_13_table_2.csv  (433 bytes)
  page_14_table_0.csv  (1,009 bytes)
  page_14_table_1.csv  (494 bytes)
  page_15_table_0.csv  (324 bytes)
  page_15_table_1.csv  (575 bytes)
  page_15_table_2.csv  (401 bytes)
  page_9_table_0.csv  (266 bytes)

🖼️  images/  (3 files)
  page_3_img_0.png  (27,513 bytes)
  page_4_img_0.png  (4

## 💬 Step 14 — Query the Pipeline

In [44]:
retrieval_pipe = RetrievalPipeline()

questions = [
    'What is the attention mechanism?',
    'What are the results shown in the tables?',
    'What is the Transformer architecture?',
    'What training data and optimiser were used?',
    'What are the main contributions of this paper?',
]

for q in questions:
    print(f'\n❓ Q: {q}')
    print(f'💡 A: {retrieval_pipe.answer(q)}')
    print('─' * 70)

retrieval_pipe.close()


❓ Q: What is the attention mechanism?


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


💡 A: diagram of a stack of data with a line of code
──────────────────────────────────────────────────────────────────────

❓ Q: What are the results shown in the tables?
💡 A: [IMAGE page 4]
──────────────────────────────────────────────────────────────────────

❓ Q: What is the Transformer architecture?
💡 A: multi - level system with multiple levels
──────────────────────────────────────────────────────────────────────

❓ Q: What training data and optimiser were used?
💡 A: [IMAGE page 4]
──────────────────────────────────────────────────────────────────────

❓ Q: What are the main contributions of this paper?
💡 A: diagram of a multi - level system with multiple levels
──────────────────────────────────────────────────────────────────────


## 🔁 Step 15 — Interactive Query Loop

In [45]:
retrieval_pipe = RetrievalPipeline()

print('Ask questions about your PDF. Type quit to stop.\n')
while True:
    q = input('❓ Your question: ').strip()
    if q.lower() in ('quit', 'exit', 'q', ''):
        break
    print(f'💡 Answer: {retrieval_pipe.answer(q)}\n')

retrieval_pipe.close()
print('Done!')

Ask questions about your PDF. Type quit to stop.

❓ Your question: encoder and decoder
💡 Answer: [IMAGE page 4]

❓ Your question: q
Done!
